In [107]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
df_train = pd.read_csv('cars_ml.csv')
df_train.head()




,brand,model,year,price,city,mileage_km,body_type,engine_volume_l,fuel_type,transmission,drive_type,steering_wheel,color,condition,generation
0,Toyota,Highlander Luxe,2017,17590000,Астана,230186.0,Кроссовер,3.5,бензин,Автомат,Полный привод,Слева,черный,NaN,2016 - 2019 3 поколение рестайлинг (U5)
1,Rox,01 VIP,2026,30490000,Алматы,NaN,Внедорожник,1.5,гибрид,Автомат,Полный привод,Слева,NaN,NaN,2023 - н.в. 1 поколение
2,Hyundai,Elantra,2020,7700000,Алматы,183426.0,Седан,2.0,бензин,Автомат,Передний привод,Слева,серый металлик,NaN,2018 - 2020 6 поколение рестайлинг (AD/ADA)
3,Kia,Sportage,2014,7000000,Петропавловск,114500.0,Кроссовер,2.0,бензин,Механика,Передний привод,Слева,серый,NaN,2010 - 2014 3 поколение (SL)
4,Toyota,Land Cruiser Prado,2014,14999999,Актобе,NaN,Внедорожник,2.7,бензин,Автомат,Полный привод,Слева,NaN,NaN,2013 - 2017 J150 рестайлинг


In [ ]:
CURRENT_YEAR = 2026
import re
def split_generation(text):
    text = str(text).lower()

    years = re.findall(r"\d{4}", text)

    start_year = int(years[0]) if len(years) >= 1 else np.nan

    if "н.в" in text:
        end_year = CURRENT_YEAR
    else:
        end_year = int(years[1]) if len(years) >= 2 else np.nan

    gen = re.search(r"(\d+)\s*покол", text)
    generation_number = int(gen.group(1)) if gen else np.nan

    is_current = 1 if "н.в" in text else 0

    return pd.Series([
        start_year,
        end_year,
        generation_number,
        is_current
    ])


def process_data(df):
    df = df.copy()
    df['price'] = np.log1p(df['price'])
    df[[
        "generation_start_year",
        "generation_end_year",
        "generation_number",
        "generation_is_current"
    ]] = df["generation"].apply(split_generation)
    df = df.drop("brand", axis=1)
    df = df.drop("generation", axis=1)
    df = df.drop("condition", axis=1)
    df = df.drop("color", axis=1)

    df['engine_volume_l'] = df['engine_volume_l'].fillna(df['engine_volume_l'].median())
    df['has_gen_number'] = df['generation_number'].apply(lambda x: 0 if pd.isna(x) else 1)
    df['mileage_km'] = df['mileage_km'].fillna(0)
    df['is_new'] = df['mileage_km'].apply(lambda x: 1 if x == 0 else 0)
    df['generation_start_year'] = df['generation_start_year'].fillna(0)
    
    df.drop("generation_is_current", axis=1, inplace=True)
    df.drop("has_gen_number", axis=1, inplace=True)
    df.drop("generation_number", axis=1, inplace=True)
    
    df = df.dropna(subset=["model", "fuel_type"])

    return df

df_train = process_data(df_train)
df_train.tail()



,model,year,price,city,mileage_km,body_type,engine_volume_l,fuel_type,transmission,drive_type,steering_wheel,generation_start_year,generation_end_year
5146,Khodro Runna,2013,13.458837,Астана,340000.0,Седан,1.6,petrol,manual,Передний привод,Слева,2012.0,2026.0
5147,Corolla,2012,15.623799,Алматы,212143.0,Седан,1.6,petrol,automatic,Передний привод,Слева,2006.0,2013.0
5148,Logan,2018,14.910784,Шымкент,366896.0,Седан,1.6,petrol-gas,manual,Передний привод,Слева,2012.0,2018.0
5149,R4,2018,15.147877,Шымкент,127279.0,Седан,1.5,petrol-gas,manual,Передний привод,Слева,2016.0,2026.0
5150,E 200,2013,15.855431,Шымкент,278691.0,Седан,2.0,petrol,automatic,Задний привод,Слева,2013.0,2017.0


In [109]:
df_train.head()

,model,year,price,city,mileage_km,body_type,engine_volume_l,fuel_type,transmission,drive_type,steering_wheel,generation_start_year,generation_end_year
0,Highlander Luxe,2017,16.682841,Астана,230186.0,Кроссовер,3.5,бензин,Автомат,Полный привод,Слева,2016.0,2019.0
1,01 VIP,2026,17.232909,Алматы,125000.0,Внедорожник,1.5,гибрид,Автомат,Полный привод,Слева,2023.0,2026.0
2,Elantra,2020,15.856731,Алматы,183426.0,Седан,2.0,бензин,Автомат,Передний привод,Слева,2018.0,2020.0
3,Sportage,2014,15.761421,Петропавловск,114500.0,Кроссовер,2.0,бензин,Механика,Передний привод,Слева,2010.0,2014.0
4,Land Cruiser Prado,2014,16.523561,Актобе,125000.0,Внедорожник,2.7,бензин,Автомат,Полный привод,Слева,2013.0,2017.0


In [95]:
import numpy as np
from catboost import CatBoostRegressor
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X,y = df_train.drop('price', axis=1), df_train['price']
cat_columns = [
    "model",
    "city",
    "body_type",
    "fuel_type",
    "transmission",
    "drive_type",
    "steering_wheel",
    
]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = CatBoostRegressor(
    iterations=10000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    loss_function="RMSE",
    random_seed=42,
    early_stopping_rounds=300,
    verbose=500
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_columns,
    eval_set=(X_test, y_test),
    use_best_model=True
)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f'MAE: {mae:.4f}')
print(f'MSE: {mse:.4f}')
print(f'R²: {r2:.4f}')  

0:	learn: 0.9878968	test: 1.0350728	best: 1.0350728 (0)	total: 4.7ms	remaining: 47s
500:	learn: 0.2925980	test: 0.3519866	best: 0.3519866 (500)	total: 872ms	remaining: 16.5s
1000:	learn: 0.2567891	test: 0.3349554	best: 0.3349554 (1000)	total: 1.85s	remaining: 16.6s
1500:	learn: 0.2376666	test: 0.3289295	best: 0.3289240 (1496)	total: 2.79s	remaining: 15.8s
2000:	learn: 0.2221913	test: 0.3239488	best: 0.3239471 (1999)	total: 3.74s	remaining: 14.9s
2500:	learn: 0.2103318	test: 0.3215948	best: 0.3215948 (2500)	total: 4.63s	remaining: 13.9s
3000:	learn: 0.1991456	test: 0.3196081	best: 0.3195726 (2973)	total: 5.57s	remaining: 13s
3500:	learn: 0.1903655	test: 0.3187278	best: 0.3185819 (3339)	total: 6.49s	remaining: 12s
4000:	learn: 0.1824010	test: 0.3176123	best: 0.3175829 (3975)	total: 7.48s	remaining: 11.2s
4500:	learn: 0.1752241	test: 0.3171218	best: 0.3171137 (4492)	total: 8.46s	remaining: 10.3s
5000:	learn: 0.1688941	test: 0.3167431	best: 0.3167004 (4972)	total: 9.39s	remaining: 9.39s
55

In [ ]:
manual_car = pd.DataFrame([{
    "model": "XV70",
    "year": 2021,
    "city": "Алматы",
    "body_type": "седан",
    "engine_volume_l": 2.5,
    "fuel_type": "бензин",
    "transmission": "Автомат",
    "drive_type": "передний привод",
    "steering_wheel": "Слева",
    "generation_start_year": 2020,
    
   
}])

manual_car = manual_car.reindex(columns=X_train.columns, fill_value=0)
for col in cat_columns:
    manual_car[col] = manual_car[col].astype(str).fillna("unknown")
pred_log = model.predict(manual_car)

pred_price = np.expm1(pred_log)

print(f"Predicted price: {pred_price[0]:,.0f} ₸")

Predicted price: 31,054,799 ₸


In [101]:
df_train.head()

,model,year,price,city,body_type,engine_volume_l,fuel_type,transmission,drive_type,steering_wheel,generation_start_year,generation_end_year,generation_number,has_gen_number
0,Highlander Luxe,2017,16.682841,Астана,Кроссовер,3.5,бензин,Автомат,Полный привод,Слева,2016.0,2019.0,3.0,1
1,01 VIP,2026,17.232909,Алматы,Внедорожник,1.5,гибрид,Автомат,Полный привод,Слева,2023.0,2026.0,1.0,1
2,Elantra,2020,15.856731,Алматы,Седан,2.0,бензин,Автомат,Передний привод,Слева,2018.0,2020.0,6.0,1
3,Sportage,2014,15.761421,Петропавловск,Кроссовер,2.0,бензин,Механика,Передний привод,Слева,2010.0,2014.0,3.0,1
4,Land Cruiser Prado,2014,16.523561,Актобе,Внедорожник,2.7,бензин,Автомат,Полный привод,Слева,2013.0,2017.0,0.0,0
